# Foco-LLM: divergência da cadeia distratora (7B)

Anexe como input a saída da versão concluída de `kaggle-ood-chain-holdout-7b`. Este notebook reutiliza seu checkpoint de 320 passos, sem novo treinamento. Requer GPU T4 e Internet ativada.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"Python 3.12 or 3.13 required; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
repository = Path("/kaggle/temp/foco-llm-divergence")
output = Path("/kaggle/working/chain-divergence-7b")
subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "feat/deduction-specialist-7b",
        "--single-branch",
        "https://github.com/Mathwesm/foco-llm.git",
        str(repository),
    ],
    check=True,
)
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repository, text=True
).strip()
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)
print("Source commit:", source_commit, flush=True)

In [ ]:
candidates = [
    path
    for path in Path("/kaggle/input").rglob("complete.json")
    if path.parent.name == "step-0320" and "ood-chain-7b" in path.parts
]
training_candidates = [
    path
    for path in Path("/kaggle/input").rglob("training-dataset.json")
    if "ood-chain-7b" in path.parts
]
if len(candidates) != 1 or len(training_candidates) != 1:
    raise RuntimeError(
        f"Expected one OOD output; adapters={len(candidates)}, "
        f"training datasets={len(training_candidates)}"
    )
adapter = candidates[0].parent
training_dataset = training_candidates[0]
command = ["poetry", "run", "python", "scripts/run_chain_divergence.py"]
subprocess.run([*command, "prepare", "--output", str(output)], cwd=repository, check=True)
dataset = output / "dataset.json"
manifest = json.loads((output / "manifest.json").read_text(encoding="utf-8"))
if (
    hashlib.sha256(dataset.read_bytes()).hexdigest() != manifest["dataset_sha256"]
    or manifest["oracle_exact"] != manifest["cases"]
):
    raise ValueError("Frozen dataset integrity check failed")
print("Dataset SHA-256:", manifest["dataset_sha256"], flush=True)

In [ ]:
subprocess.run(
    [*command, "evaluate", "--dataset", str(dataset), "--output", str(output / "base")],
    cwd=repository,
    check=True,
)
subprocess.run(
    [
        *command,
        "evaluate",
        "--dataset",
        str(dataset),
        "--output",
        str(output / "specialist"),
        "--adapter",
        str(adapter),
        "--training-dataset",
        str(training_dataset),
    ],
    cwd=repository,
    check=True,
)


def read_one(root):
    """Read the unique completed summary for an evaluation arm."""
    paths = list(root.glob("*/summary.json"))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one summary in {root}; found {len(paths)}")
    return json.loads(paths[0].read_text(encoding="utf-8"))


comparison = {
    "protocol": "chain-divergence-v1",
    "source_commit": source_commit,
    "dataset_manifest": manifest,
    "base": read_one(output / "base"),
    "deduction_specialist": read_one(output / "specialist"),
}
(output / "comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps(comparison, indent=2), flush=True)